In [16]:
!pip install requests beautifulsoup4 nltk

  Obtaining dependency information for nltk from https://files.pythonhosted.org/packages/60/90/81ac364ef94209c100e12579629dc92bf7a709a84af32f8c551b02c07e94/nltk-3.9.2-py3-none-any.whl.metadata
  Obtaining dependency information for click from https://files.pythonhosted.org/packages/98/78/01c019cdb5d6498122777c1a43056ebb3ebfeef2076d9d026bfe15583b2b/click-8.3.1-py3-none-any.whl.metadata
  Obtaining dependency information for joblib from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for regex>=2021.8.3 from https://files.pythonhosted.org/packages/90/b0/7c2a74e74ef2a7c32de724658a69a862880e3e4155cba992ba04d1c70400/regex-2026.1.15-cp312-cp312-macosx_10_13_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.1 MB/s eta 0:00:00
  Obtaining dependency information for tqdm from https://files.pythonhosted.org/packages/16/e1/3079a9ff9b8e

## 1. Generar una lista con las URLs que contendrán las letras de las canciones.

In [3]:
import requests
url_base = 'https://www.letras.com'
band = 'carpenters'
url_get = requests.get(url_base + '/' + band + '/') 

In [4]:
from bs4 import BeautifulSoup
html = BeautifulSoup(url_get.content, 'html.parser') 

In [5]:
lyrics = [i.get('data-shareurl')
 for i in html.find_all('li', class_='songList-table-row')]
lyrics = list(set(lyrics))
print(lyrics)
print(len(lyrics))

['https://www.letras.com/carpenters/441255/', 'https://www.letras.com/carpenters/441308/', 'https://www.letras.com/carpenters/441567/', 'https://www.letras.com/carpenters/7026/', 'https://www.letras.com/carpenters/441411/', 'https://www.letras.com/carpenters/98625/', 'https://www.letras.com/carpenters/359896/', 'https://www.letras.com/carpenters/94718/', 'https://www.letras.com/carpenters/440200/', 'https://www.letras.com/carpenters/440559/', 'https://www.letras.com/carpenters/441261/', 'https://www.letras.com/carpenters/1338810/', 'https://www.letras.com/carpenters/441294/', 'https://www.letras.com/carpenters/441687/', 'https://www.letras.com/carpenters/293214/', 'https://www.letras.com/carpenters/65575/', 'https://www.letras.com/carpenters/98622/', 'https://www.letras.com/carpenters/94720/', 'https://www.letras.com/carpenters/100792/', 'https://www.letras.com/carpenters/7016/', 'https://www.letras.com/carpenters/293203/', 'https://www.letras.com/carpenters/293215/', 'https://www.letr

## 2. Crear el corpus con el texto de las letras de canciones

In [ ]:
corpus = []

for url in lyrics: 
    response = requests.get(url)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Buscamos el contenedor principal de la letra
    lyric_div = soup.find('div', class_='lyric-original')
    
    if not lyric_div:
        print(f"No se encontró lyric-original en {url}")
        continue
    
    # Extraemos todo el texto usando \n como separador → respeta los <br>
    full_text = lyric_div.get_text(separator='\n', strip=True)
    
    # Dividimos en líneas y limpiamos
    verses = []
    for line in full_text.split('\n'):
        cleaned = line.strip()
        if cleaned: 
            if 'viewFractions' not in cleaned and 'data-event' not in cleaned:
                verses.append(cleaned)
    
    if verses:
        corpus.append(verses)
        print(f"Letra extraída ({len(verses)} versos) de {url}")
    else:
        print(f"No se extrajeron versos de {url}")

# Guardar en JSON
if corpus:
    with open('letras_corpus.json', 'w', encoding='utf-8') as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)
    
    print(f"\nGuardado correctamente en 'letras_corpus.json'")
    print(f"Total de canciones: {len(corpus)}")
else:
    print("No se extrajo ninguna letra → no se creó el archivo")

Letra extraída (33 versos) de https://www.letras.com/carpenters/441255/
Letra extraída (20 versos) de https://www.letras.com/carpenters/441308/
Letra extraída (7 versos) de https://www.letras.com/carpenters/441567/
Letra extraída (42 versos) de https://www.letras.com/carpenters/7026/
Letra extraída (25 versos) de https://www.letras.com/carpenters/441411/
Letra extraída (27 versos) de https://www.letras.com/carpenters/98625/
Letra extraída (42 versos) de https://www.letras.com/carpenters/359896/
Letra extraída (24 versos) de https://www.letras.com/carpenters/94718/
Letra extraída (29 versos) de https://www.letras.com/carpenters/440200/
Letra extraída (41 versos) de https://www.letras.com/carpenters/440559/
Letra extraída (22 versos) de https://www.letras.com/carpenters/441261/
Letra extraída (15 versos) de https://www.letras.com/carpenters/1338810/
Letra extraída (27 versos) de https://www.letras.com/carpenters/441294/
Letra extraída (28 versos) de https://www.letras.com/carpenters/4416

## 3. Generar n-gramas para el corpus construido

In [20]:
import json
from nltk.tokenize import word_tokenize
import nltk

# Asegurarse de tener el tokenizador de NLTK (solo la primera vez)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


True

In [ ]:
# Cargar el corpus desde el JSON
with open('letras_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)
tokenized_corpus = []

for cancion_versos in corpus:
    for linea in cancion_versos:
        tokens = word_tokenize(linea.lower())
        if tokens:
            tokenized_corpus.append(tokens)

with open('tokenized_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(tokenized_corpus, f, ensure_ascii=False, indent=2)

In [25]:
tokenized_corpus[:3]

[['i', 'ran', 'away', 'from', 'you'],
 ['and', 'left', 'you', 'crying'],
 ['and', 'though', 'i', "'m", 'back', 'to', 'stay']]

In [29]:
from nltk.util import ngrams

with open('tokenized_corpus.json', 'r', encoding='utf-8') as f:
    flat_tokenized = json.load(f)

ngrams_corpus = []
for verso_tokens in flat_tokenized:
    if len(verso_tokens) >= 3:
        trigramas_verso = list(ngrams(verso_tokens, 3))
        ngrams_corpus.append(trigramas_verso)

with open('ngrams_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(ngrams_corpus, f, ensure_ascii=False, indent=2)

In [30]:
ngrams_corpus[:3] 

[[('i', 'ran', 'away'), ('ran', 'away', 'from'), ('away', 'from', 'you')],
 [('and', 'left', 'you'), ('left', 'you', 'crying')],
 [('and', 'though', 'i'),
  ('though', 'i', "'m"),
  ('i', "'m", 'back'),
  ("'m", 'back', 'to'),
  ('back', 'to', 'stay')]]

## 4. Construir un modelo de cadena de Markov

In [4]:
from collections import defaultdict
import json

def cargar_y_construir_modelo(ruta_json='ngrams_corpus.json'):
    with open(ruta_json, 'r', encoding='utf-8') as f:
        datos = json.load(f)
    
    modelo = defaultdict(list)
    
    for lista_trigramas in datos:
        for trigrama in lista_trigramas:
            if len(trigrama) == 3:
                w1, w2, w3 = trigrama
                estado = (w1, w2)
                modelo[estado].append(w3)
    
    return modelo

modelo = cargar_y_construir_modelo('ngrams_corpus.json')

In [5]:
# Función para generar una frase
import random

def generate_sentence(model, initial_word, num_words):
    # Crear una lista de pares de palabras que incluyan la palabra inicial
    pairs = [estado for estado in model.keys() if initial_word in estado]

    # Si no hay pares que incluyan la palabra inicial, lanza una excepción
    if not pairs:
        raise ValueError(f"No se encontraron pares que incluyan la palabra '{initial_word}'")

    # Selecciona un par de palabras aleatorio de la lista de pares

    estado_actual = random.choice(pairs)
    # Convierte el par de palabras seleccionado en una lista, y asígnalo añádelo
    # a la lista de palabras que constituye la frase que estás generando
    sentence = list(estado_actual)

    # Generaremos las siguientes palabras iterando ‘num_words-1’
    for _ in range(num_words - 1):
        try:
            # Intenta seleccionar una palabra aleatoria de las siguientes posibles
            # basándose en el último par de palabras
            siguiente_palabra = random.choice(model[estado_actual])
            # Agrega la palabra seleccionada a la frase
            sentence.append(siguiente_palabra)
            # Actualiza el estado actual con las dos últimas palabras
            estado_actual = (estado_actual[1], siguiente_palabra)
        except IndexError:
            # Si no hay palabras para continuar, detiene la generación
            break
    # Une las palabras de la frase en una cadena y devuelve la frase completa
    return ' '.join(sentence)

In [8]:
print(generate_sentence(modelo, 'no', 15)) 

n't no fool for love and get love returned


### Según este modelo de cadenas de Markov, ¿de qué depende la probabilidad de la siguiente palabra?

Según este modelo de cadenas de Markov que hemos construido: La probabilidad de la siguiente palabra **depende exclusivamente de las dos palabras anteriores**.


### ¿Solo del n-grama actual, o de las palabras anteriores?

Solo del n-grama actual.
En el modelo de Markov que hemos construido y que estamos usando (orden 2), la probabilidad de la siguiente palabra depende exclusivamente del par de palabras inmediatamente anteriores (el bigrama actual).

### ¿Por qué?

Porque así lo decidimos al construir el modelo: elegimos una cadena de Markov de orden 2 y por definición matemática de Markov, el futuro solo puede depender del estado presente y aquí el “estado presente” son las últimas dos palabras.

### La implementación que se ha realizado está basada en un mapa o diccionario <clave, valor> de <bigrama, lista_palabras>. ¿Se te ocurre otra implementación?

Podemos usar la estructura defaultdict(Counter) que la clave es (w1, w2) y el valor es Counter({'w3': 15, 'w4': 7, 'w5': 2}).

### ¿Cómo afectaría a tu implementación? 

- La generación será notablemente más fiel al corpus
- Ahorro importante de memoria y disco
- Más fácil experimentar después
- Pequeño costo

### ¿Qué ventajas y desventajas tendría cada una de las aproximaciones?

1. defaultdict(list)
Ventajas: código más corto, más fácil de entender, generación muy rápida.
Desventajas: gasta mucha memoria con repeticiones, generación no ponderada correctamente (todas las ocurrencias igual de probables), difícil ver frecuencias reales.
2. defaultdict(Counter)
Ventajas: ahorra mucha memoria, generación ponderada correctamente (más realista), fácil ver las palabras más frecuentes, fácil pasar a probabilidades.
Desventajas: generación un poco más lenta, código de generación algo más largo.
3. Probabilidades precalculadas
Ventajas: generación más rápida posible, muy flexible para temperatura/top-k/etc.
Desventajas: paso extra de normalización, pierdes los conteos absolutos, algo más memoria que Counter en algunos casos.



# 5.Generar toda la letra de una canción

## 6.Usar la implementación de modelos de lenguaje de NLKT.